In [1]:
!pip install openpyxl pandas matplotlib rdflib


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import XSD, RDFS
import re

### Knowledge Graph Triplet Construction - 

The selected format is RDFTurtle, with Python's RDFLib, for easier integration. The type annotations and construction was also simpler for us to understand, and we make use of the defined **ontology.ttl** file in order to generate the **data.ttl** file.

We decided to have two types of nodes:
- **Neighborhood Node** - created for each of the 116 Eindhoven neighborhoods, keeping only the neighborhood name as a literal.
- **Observation Node** - created for each unique (neighborhood, year) pair, keeping all indicator values as typed datatype literals.

The two nodes are linked bi-directionally, and it is important to note that NaN values are silently skipped during creation, consistent with the decision throughout the pipeline to preserve structural missingness rather than interpolate misleading values.

In [3]:
EIN = Namespace("http://eindhoven.nl/livability/")
g = Graph()
g.bind("ein", EIN)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)

# Reading merged data.
df = pd.read_csv("../data/processed/merged.csv", index_col=0)

PROP_MAP = {
    "livability_score":                  EIN.livabilityScore,
    "livability_deviation_from_avg":     EIN.livabilityDeviation,
    "physical_environment_deviation":    EIN.physicalEnvironmentDeviation,
    "nuisance_insecurity_deviation":     EIN.nuisanceInsecurityDeviation,
    "social_cohesion_deviation":         EIN.socialCohesionDeviation,
    "facilities_deviation":              EIN.facilitiesDeviation,
    "housing_stock_deviation":           EIN.housingStockDeviation,
    "dist_supermarket":                  EIN.distSupermarket,
    "dist_daily_groceries":              EIN.distDailyGroceries,
    "dist_attraction":                   EIN.distAttraction,
    "dist_library":                      EIN.distLibrary,
    "dist_museum":                       EIN.distMuseum,
    "dist_performing_arts":              EIN.distPerformingArts,
    "dist_cinema":                       EIN.distCinema,
    "dist_sauna":                        EIN.distSauna,
    "dist_cafe":                         EIN.distCafe,
    "dist_cafeteria":                    EIN.distCafeteria,
    "dist_restaurant":                   EIN.distRestaurant,
    "dist_swimming_pool":                EIN.distSwimmingPool,
    "dist_ice_rink":                     EIN.distIceRink,
    "dist_gp_post":                      EIN.distGpPost,
    "dist_gp_practice":                  EIN.distGpPractice,
    "dist_pharmacy":                     EIN.distPharmacy,
    "dist_hospital":                     EIN.distHospital,
    "dist_daycare":                      EIN.distDaycare,
    "dist_after_school_care":            EIN.distAfterSchoolCare,
    "dist_primary_school":               EIN.distPrimarySchool,
    "dist_secondary_school":             EIN.distSecondarySchool,
    "dist_highway_ramp":                 EIN.distHighwayRamp,
    "dist_train_station":                EIN.distTrainStation,
    "rental_dwellings":                  EIN.rentalDwellings,
    "owner_occupied_dwellings":          EIN.ownerOccupiedDwellings,
    "corporation_dwellings":             EIN.corporationDwellings,
    "total_dwellings":                   EIN.totalDwellings,
    "pct_occupied":                      EIN.pctOccupied,
    "avg_woz_value":                     EIN.avgWozValue,
    "pct_feels_unsafe":                  EIN.pctFeelsUnsafe,
    "pct_perceives_crime":               EIN.pctPerceivesCrime,
    "safety_rating":                     EIN.safetyRating,
    "pct_noise_air_traffic":             EIN.pctNoiseAirTraffic,
    "pct_noise_rail_traffic":            EIN.pctNoiseRailTraffic,
    "pct_noise_business":                EIN.pctNoiseBusiness,
    "pct_noise_road_traffic":            EIN.pctNoiseRoadTraffic,
    "pct_noise_events":                  EIN.pctNoiseEvents,
    "pct_noise_neighbors":               EIN.pctNoiseNeighbors,
    "pct_polluted_air":                  EIN.pctPollutedAir,
    "pct_bad_smell":                     EIN.pctBadSmell,
    "pct_any_noise":                     EIN.pctAnyNoise,
    "address_density":                   EIN.addressDensity,
    "urbanity_class":                    EIN.urbanityClass,
    "area_water_ha":                     EIN.areaWaterHa,
    "area_land_ha":                      EIN.areaLandHa,
    "area_total_ha":                     EIN.areaTotalHa,
    "cohesion_scale_score":              EIN.cohesionScaleScore,
    "pct_actively_engaged":              EIN.pctActivelyEngaged,
    "pct_wants_to_engage":               EIN.pctWantsToEngage,
    "pct_maybe_engage":                  EIN.pctMaybeEngage,
    "population":                        EIN.population,
    "dimension_scores_available":        EIN.dimensionScoresAvailable,
    "survey_available":                  EIN.surveyAvailable,
    "engagement_available":              EIN.engagementAvailable,
    "woz_available":                     EIN.wozAvailable,
}

# Integer columns need to be specified as they use xsd:integer instead of xsd:decimal.
INT_COLS = {
    "population", "total_dwellings", "address_density"
}

# Likewise we identify string columns.
STR_COLS = {"urbanity_class"}

# Likewise we identify boolean columns.
BOOL_COLS = {
    "dimension_scores_available", "survey_available", "engagement_available", "woz_available"
}

# We need to take the neighborhood name and ensure that it is cleaned, safe for a URI.
def clean_uri(s):
    return re.sub(r"[^a-zA-Z0-9_]", "_", str(s).strip())

for col in INT_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in PROP_MAP.keys():
    if col in BOOL_COLS or col in INT_COLS or col in STR_COLS:
        continue
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [4]:
neighborhoods_seen = set()

for _, row in df.iterrows():
    nb_name = str(row["neighborhood"]).strip()
    year = int(row["year"])
    nb_safe = clean_uri(nb_name)
    obs_id = f"{nb_safe}_{year}"

    nb_uri = EIN[nb_safe]
    obs_uri = EIN[obs_id]

    # Neighborhood node is added once.
    if nb_safe not in neighborhoods_seen:
        g.add((nb_uri, RDF.type, EIN.Neighborhood))
        g.add((nb_uri, EIN.hasName, Literal(nb_name, datatype=XSD.string)))
        neighborhoods_seen.add(nb_safe)

    # Observation nodes per (neighborhood, year)
    g.add((obs_uri, RDF.type, EIN.Observation))
    g.add((obs_uri, EIN.inNeighborhood, nb_uri))
    g.add((obs_uri, EIN.inYear, Literal(year, datatype=XSD.integer)))

    # Create the link between neighborhood and observation
    g.add((nb_uri, EIN.hasObservation, obs_uri))

    # Add all indicator properties, skipping NaN
    for col, prop in PROP_MAP.items():
        val = row.get(col)
        if pd.isna(val):
            continue
        
        if col in BOOL_COLS:
            dtype = XSD.boolean
        elif col in INT_COLS:
            dtype = XSD.integer
        elif col in STR_COLS:
            dtype = XSD.string
        else:
            dtype = XSD.decimal

        g.add((obs_uri, prop, Literal(val, datatype=dtype)))


g.serialize("../data/processed/data.ttl", format="turtle")

<Graph identifier=Na13b0ec3277640d78a09a66185fc432f (<class 'rdflib.graph.Graph'>)>